## Repaso

In [1]:
import csv
from collections import defaultdict
import copy
from prettytable import PrettyTable

### Ejercicio 7 (Nivel 4): reto analítico

**Objetivo**: con los datos limpios, comparar el desempeño de enero frente a febrero.\
**Resultado requerido**: una tabla simple impresa (país, total enero, total febrero) y una interpretación escrita de 4 a 5 frases que incluya:
- Qué patrón ves.
- Qué limitaciones tienen estos datos, incluyendo el efecto de las filas que descartaste.
- Qué decisión de negocio podría apoyar.

**Restricción**: decide tú cómo obtener el mes a partir de la fecha y justifica tu decisión en una línea.

In [2]:
# Leer archivo csv
with open("data/ventas_sucias.csv", encoding="utf-8", newline="") as data_file:
    reader = csv.DictReader(data_file)
    data = list(reader)

In [3]:
# Crear una copia de la data original para trabajar sobre la copia.
data_copy = copy.deepcopy(data)

In [4]:
# Corregir nombres de países que tengan espacios en blanco y no tengan primer letra en mayúscula
for row in data_copy:
    row['pais'] = row['pais'].title().strip()

In [5]:
unrecoverable_rows = []

# Convertir montos recuperables a decimal. Si el monto tiene símbolo $, se elimina el símbolo y se convierte.
for row in data_copy:
    try:
        row['monto'] = float(row['monto'].replace('$', '').strip())
    except ValueError:
        unrecoverable_rows.append(row)

In [6]:
print(f"Unrecoverable rows:")
for row in unrecoverable_rows:
    print(row)

Unrecoverable rows:
{'id_venta': '5', 'cliente': 'Sofía', 'pais': 'México', 'fecha': '2026-01-15', 'monto': ''}
{'id_venta': '9', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-02-14', 'monto': 'abc'}


In [7]:
# Agregar columna "mes" al dataset para poder usarla como guía para la sumatoria de totales por mes de cada país.
# Los meses quedan identificados por un string con el número de mes. 
# Se usa split() por simplicidad porque en este caso el dataset es pequeño, fácil de visualizar y todas la fechas tienen el mismo formato.
for row in data_copy:
    row['mes'] = row['fecha'].split('-')[1]
    print(row)

{'id_venta': '1', 'cliente': 'Ana', 'pais': 'Guatemala', 'fecha': '2026-01-05', 'monto': 120.5, 'mes': '01'}
{'id_venta': '2', 'cliente': 'Luis', 'pais': 'México', 'fecha': '2026-01-07', 'monto': 80.0, 'mes': '01'}
{'id_venta': '3', 'cliente': 'Marta', 'pais': 'Guatemala', 'fecha': '2026-01-09', 'monto': 45.25, 'mes': '01'}
{'id_venta': '4', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-01-12', 'monto': 200.0, 'mes': '01'}
{'id_venta': '5', 'cliente': 'Sofía', 'pais': 'México', 'fecha': '2026-01-15', 'monto': '', 'mes': '01'}
{'id_venta': '6', 'cliente': 'Diego', 'pais': 'Guatemala', 'fecha': '2026-01-20', 'monto': 99.99, 'mes': '01'}
{'id_venta': '7', 'cliente': 'Ana', 'pais': 'Guatemala', 'fecha': '2026-02-02', 'monto': 75.0, 'mes': '02'}
{'id_venta': '8', 'cliente': 'Luis', 'pais': 'México', 'fecha': '2026-02-10', 'monto': 130.0, 'mes': '02'}
{'id_venta': '9', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-02-14', 'monto': 'abc', 'mes': '02'}
{'id_venta': '10', 'cli

In [8]:
# Verificar los meses y paises presentes en el dataset
months = set([row['mes'] for row in data_copy])
countries = set([row['pais'] for row in data_copy])
print(months)
print(countries)

{'01', '02'}
{'Colombia', 'Guatemala', 'México'}


In [9]:
jan_dates = [data['fecha'] for data in data_copy if data['mes'] == '01']
feb_dates = [data['fecha'] for data in data_copy if data['mes'] == '02']

print(f"Enero: fecha_min: {min(jan_dates)}, fecha_max: {max(jan_dates)}")
print(f"Febrero: fecha_min: {min(feb_dates)}, fecha_max: {max(feb_dates)}")

Enero: fecha_min: 2026-01-05, fecha_max: 2026-01-20
Febrero: fecha_min: 2026-02-02, fecha_max: 2026-02-18


In [10]:
# Descartar filas irrecuperables por montos inválidos
for row in data_copy[:]:
    if not isinstance(row['monto'], float):
        data_copy.remove(row)

In [11]:
count = defaultdict(lambda: defaultdict(float))

# Conteo de cantidad de ventas por mes y país
for row in data_copy:
    month = row['mes']
    if month in months:
        count[month][row['pais']] += 1

count = {m: dict(countries_count) for m, countries_count in count.items()}
print(count)

{'01': {'Guatemala': 3.0, 'México': 1.0, 'Colombia': 1.0}, '02': {'Guatemala': 2.0, 'México': 1.0}}


In [12]:
totals = defaultdict(lambda: defaultdict(float))

# Calcular totales por país para cada mes
for row in data_copy:
    month = row['mes']
    if month in months:
        totals[month][row['pais']] += row['monto']

totals = {month: dict(countries_total) for month, countries_total in totals.items()}
print(totals)

{'01': {'Guatemala': 265.74, 'México': 80.0, 'Colombia': 200.0}, '02': {'Guatemala': 135.0, 'México': 130.0}}


In [13]:
# Resultados
table = PrettyTable()

table.field_names = ['pais', 'total enero', 'total febrero']
# Agregar totales a la tabla
for country in sorted(countries):
    table.add_row([country, totals['01'].get(country), totals['02'].get(country)])

print(table)

+-----------+-------------+---------------+
|    pais   | total enero | total febrero |
+-----------+-------------+---------------+
|  Colombia |    200.0    |      None     |
| Guatemala |    265.74   |     135.0     |
|   México  |     80.0    |     130.0     |
+-----------+-------------+---------------+


**Conclusión**\
Guatemala presenta una reducción del 49.20% en su total de ventas de enero a febrero, dato obtenido de ; los totales de ventas de México en enero y Colombia en febrero están incompletos debido a que se excluyeron registros de venta con montos inválidos.

Si los registros de venta inválidos corresponden a ventas reales con montos positivos, su exclusión provoca una subestimación de los ingresos para México y Colombia en los meses correspondientes y debido a los datos faltantes no se puede cuantificar la magnitud de la subestimación.

Los totales de venta de México y Colombia no son concluyentes por lo que no son datos fiables para determinar si las ventas aumentaron o disminuyeron. Se recomienda recuperar el monto faltante de las ventas descartadas para evaluar el desempeño real. 

Con estos datos, la gerencia podría considerar estrategias comerciales para buscar incrementar la ventas de Guatemala debido a su reducción de enero a febrero. Esto no aplica a México y Colombia ya que para estos paises es conveniente esperar a tener datos fiables que permitan medir el desempeño real antes de tomar decisiones.
